# 2 - State and action design

The full game state is astronomically large - a hand of three resource types per player, decks,
tools, volcano progress, and so on, easily exceeding $10^{50}$ configurations. A tabular agent
cannot key a value table on that. The core modelling work is therefore to project the state onto a
small, decision-relevant abstraction. This notebook lays out those abstractions and checks that the
number of states that actually occur in play is small enough for a Q-table to fill in.

In [1]:
import sys
import pathlib
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the repository root (the directory that contains simulation_engine) so the
# notebook runs regardless of the directory Jupyter was launched from.
repository_path = pathlib.Path.cwd()
while not (repository_path / "simulation_engine").is_dir() and repository_path != repository_path.parent:
    repository_path = repository_path.parent
machine_learning_path = repository_path / "notebooks" / "machine_learning"
for path_entry in (str(repository_path), str(machine_learning_path)):
    if path_entry not in sys.path:
        sys.path.insert(0, path_entry)

import simulation_engine as engine
import simulation_engine.rl as rl
import helpers.rl_plots as rl_plots

%matplotlib inline
%load_ext autoreload
%autoreload 2

## Mission-selection state (7 features)

For *which mission to attempt*, only a handful of facts matter. `MissionStateEncoder` maps a state
to a 7-tuple of small integers:

| # | feature | values |
|---|---------|--------|
| 0 | volcano urgency bucket of cards remaining | 4 |
| 1 | boat parts already built (clamped 0-5) | 6 |
| 2 | a Panic card is pending | 2 |
| 3 | next-needed boat part: absent / present-unaffordable / present-affordable | 3 |
| 4 | any boat mission affordable now | 2 |
| 5 | bitmask of non-boat categories on offer {fire, food, shelter} | 8 |
| 6 | active player can individually afford some offered mission | 2 |

The product is $4 \times 6 \times 2 \times 3 \times 2 \times 8 \times 2 = 4608$ possible keys.

In [2]:
mission_encoder = rl.MissionStateEncoder()
participant_encoder = rl.ParticipantStateEncoder()
print('mission-state keys:', mission_encoder.cardinality())
print('participant-state keys:', participant_encoder.cardinality())
print('mission action categories:', rl.MISSION_ACTION_CARDINALITY)
print('participant actions:', rl.PARTICIPANT_ACTION_CARDINALITY)

mission-state keys: 4608
participant-state keys: 144
mission action categories: 5
participant actions: 2


### Why semantic action categories, not slot indices

The three active missions sit in slots that hold different missions every round, so a value keyed on
"slot 0" would not generalise. Instead the mission action is one of five **semantic categories**:
next-needed boat part, other boat part, fire, food, or shelter. The same category means the same
kind of choice in every state. At decision time we list the legal missions, the agent picks a
category, and a deterministic tie-break (boat build order, then mission definition order) resolves
the concrete mission.

In [3]:
from simulation_engine.initialization import init_game
from simulation_engine.models import MissionName

random.seed(11)
state = init_game(player_count = 7)
active_player = state.players[state.active_player_index]

legal = rl.legal_missions(active_player, state)
print('active missions:', [mission.value for mission in state.active_missions])
print('legal candidates:', [mission.value for mission in legal])
print('mission state key:', mission_encoder.encode(active_player, state))
print()
for mission_name in legal:
    category = rl.encode_mission_action(mission_name, state)
    print(f'  {mission_name.value:20s} -> category {category}')
print('decode category 0 (next boat) ->', rl.decode_mission_action(0, legal, state))

active missions: ['torch_for_the_night', 'fetch_water', 'hunt']
legal candidates: ['hunt']
mission state key: (2, 0, 0, 0, 0, 3, 1)

  hunt                 -> category 3
decode category 0 (next boat) -> None


## Participant-selection state (6 features)

Staffing is a per-candidate decision. For each eligible player `ParticipantStateEncoder` produces a
6-tuple: affordability (cannot/exact/surplus), whether the character's ability is active on this
mission, whether they are the Craftsman while a tool needs repair, their score lead over the active
player (behind/slightly/far ahead), whether the mission is a boat, and whether they are the active
player. That is $3 \times 2 \times 2 \times 3 \times 2 \times 2 = 144$ keys. The agent learns an
*include* value per candidate and the policy takes the top `players_count` by margin - a learned
version of the hand-tuned scoring heuristic the engine ships with.

In [4]:
from simulation_engine.models import Mission
from simulation_engine.agents.feasibility import player_afford_level

mission = Mission.get(state.active_missions[0])
rows = []
for player in state.players:
    key = participant_encoder.encode(player, active_player, mission, state)
    rows.append({
        'character': player.character.value,
        'afford_level': player_afford_level(player, mission, state).value,
        'state_key': key,
    })
pd.DataFrame(rows)

,character,afford_level,state_key
0,cook,cannot_afford,"(0, 0, 0, 0, 0, 1)"
1,fire_starter,cannot_afford,"(0, 1, 0, 0, 0, 0)"
2,sailor,cannot_afford,"(0, 0, 0, 0, 0, 0)"
3,gatherer,surplus,"(2, 0, 0, 0, 0, 0)"
4,cook,cannot_afford,"(0, 0, 0, 0, 0, 0)"
5,builder,surplus,"(2, 0, 0, 0, 0, 0)"
6,craftsman,cannot_afford,"(0, 0, 0, 0, 0, 0)"


## How many states actually occur?

A cardinality of 4608 is an upper bound; most combinations never arise (for example, 5 boat parts
built only happens at 8 players). We play many baseline games across all player counts and record
every mission state the engine encounters. The observing selector below records the state and then
defers to the rule-based `vote_for_mission`, so play is identical to the baseline.

In [5]:
from simulation_engine.agents import vote_for_mission

visited_mission_states = set()

def observing_selector(active_player, state):
    visited_mission_states.add(mission_encoder.encode(active_player, state))
    return vote_for_mission(active_player, state)

for player_count in (6, 7, 8):
    engine.run_scenario(
        player_count = player_count, n_games = 600, base_seed = 42,
        mission_selector = observing_selector,
    )

print(f'distinct mission states seen in 1800 baseline games: {len(visited_mission_states)}')
print(f'of {mission_encoder.cardinality()} possible ({len(visited_mission_states) / mission_encoder.cardinality():.1%} of the space)')

distinct mission states seen in 1800 baseline games: 789
of 4608 possible (17.1% of the space)


Only a few hundred mission states occur in practice, so a sparse dictionary-backed Q-table covers
the reachable space densely with thousands of self-play games. Notebook 3 puts that to work.